In [ ]:
import time

import numpy as np
import pandas as pd
from scipy.special import expit

from Pipeline.Model.ABC_ELM import ABC_ELM2
from Pipeline.Model.CrossValidationDataSplit import CrossValidationDataSplit
from Pipeline.Model.EvaluationMatrix import EvaluationMatrix

In [ ]:
filePath = '../../Dataset/UCI_Gallstone_Dataset.csv'
df = pd.read_csv(filePath)

targetCol = ['Gallstone Status']
X = df.drop(targetCol, axis=1)
y = df[targetCol]
features_size = X.shape[1]

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [ ]:
def stable_sigmoid(x):
    # Numerically stable activation to prevent float64 overflow in randomized spaces
    return expit(x)

class AcademicEvaluation:
    @staticmethod
    def rigorous_metaheuristic_validation(X, y,
                                          k_folds=5,
                                          seeds_per_fold=10,
                                          hidden_size=32,
                                          reg_lambda=1.0,
                                          algo_type='algo3'):
        """
        Executes a nested cross-validation to isolate data variance from algorithmic variance.
        """
        print(f"[*] Initializing Rigorous Evaluation: {k_folds}-Fold CV with {seeds_per_fold} Seeds/Fold.")

        # 1. Isolate Data Variance
        # We use Stratified K-Fold to ensure class distributions remain consistent across splits.
        splitter = CrossValidationDataSplit(random_state=42, test_size=0.2, k_fold=k_folds)
        x_test, y_test, folds = splitter.k_fold_data_spiting(X, y)

        global_results = []

        for fold_idx in range(k_folds):
            print(f"\n[+] Processing Fold {fold_idx + 1}/{k_folds}")
            fold = folds[fold_idx]

            # Data is already scaled by DataSplit.py inside the fold to prevent data leakage
            x_train, y_train = fold['X_train_fold'].values, fold['y_train_fold'].values
            x_val, y_val     = fold['X_val_fold'].values, fold['y_val_fold'].values

            fold_metrics_accumulator = []

            # 2. Isolate Algorithmic Variance
            # We iterate through predefined seeds to quantify the stability of the swarm convergence.
            for seed in range(seeds_per_fold):

                # We enforce iter_max=100 as per the established literature baseline[cite: 338].
                abc_model = ABC_ELM2(
                    feature_size=x_train.shape[1],
                    hidden_size=hidden_size,
                    activation_function=stable_sigmoid,
                    regularization_lambda=reg_lambda,
                    algo_type=algo_type,
                    random_seed=seed,
                    SN=10, limit=10, iter_max=100
                )

                abc_model.fit(x_train, y_train)
                y_pred = abc_model.predict(x_val)

                # Extract metrics for this specific stochastic trajectory
                metrics = EvaluationMatrix(y_val, y_pred).get_all_metrics()

                # Append seed identifier for granular tracking
                metrics['Fold'] = fold_idx
                metrics['Seed'] = seed
                fold_metrics_accumulator.append(metrics)

            # Aggregate intra-fold metrics to observe swarm stability
            fold_df = pd.DataFrame(fold_metrics_accumulator)

            fold_summary = {
                'Fold': fold_idx,
                'Mean_Accuracy': fold_df['Accuracy'].mean(),
                'StdDev_Accuracy': fold_df['Accuracy'].std(),
                'Mean_F1': fold_df['F1-Score'].mean(),
                'StdDev_F1': fold_df['F1-Score'].std(),
                'Mean_MCC': fold_df['MCC'].mean(),
            }
            global_results.append(fold_summary)
            print(f"    -> Fold {fold_idx} Mean Accuracy: {fold_summary['Mean_Accuracy']:.4f} (±{fold_summary['StdDev_Accuracy']:.4f})")

        # 3. Final Statistical Aggregation
        final_matrix = pd.DataFrame(global_results)
        return final_matrix, pd.DataFrame(fold_metrics_accumulator)

# ==========================================
# Execution Block (Testing the Architecture)
# ==========================================
if __name__ == "__main__":
    # Simulate loading the Gallstone Dataset (Replace with actual pd.read_csv)
    # Using dummy data for structural testing
    print("[*] Generating synthetic dataset for pipeline verification...")
    np.random.seed(42)
    X_dummy = pd.DataFrame(np.random.rand(200, 15))
    y_dummy = pd.Series(np.random.randint(0, 2, 200))

    # Execute the rigorous validation
    start_time = time.time()
    summary_df, granular_df = AcademicEvaluation.rigorous_metaheuristic_validation(
        X=X_dummy,
        y=y_dummy,
        k_folds=5,           # 5 Data splits
        seeds_per_fold=10,   # 10 Swarm initializations per split (Reduce from 50 for initial testing)
        hidden_size=32,
        reg_lambda=1.223,
        algo_type='algo3'
    )

    print(f"\n[!] Global Evaluation Completed in {time.time() - start_time:.2f} seconds.")
    print("\n=== Cross-Fold Statistical Summary ===")
    print(summary_df.to_string(index=False))